In [ ]:
# !pip install transformers


In [3]:
from transformers import pipeline

In [ ]:
# ① 감성 분석 (영어)
sentiment = pipeline("sentiment-analysis")
result = sentiment("I love this course!")
# result = sentiment("I don't like this course!")
# result = sentiment("영화 엔드게임 재미 없어요") # 한글로 학습이 안됨 -> 영어로 학습
print(result)  # [{'label': 'POSITIVE', 'score': 0.9998}]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.7535268068313599}]


In [12]:
# ② 텍스트 분류 (한국어 - 사전학습된 모델 사용)
classifier = pipeline(
    "text-classification",
    model="snunlp/KR-FinBert-SC"  # 한국어 금융 감성분석
)
# result = classifier("이 주식은 앞으로 크게 오를 것 같습니다.")
result = classifier("이 주식은 앞으로 크게 내릴 것 같습니다.")
print(result)  # [{'label': 'positive', 'score': 0.92}]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'neutral', 'score': 0.9995540976524353}]


In [13]:
# ③ 개체명 인식(NER)
ner = pipeline("ner", model="Leo97/KoELECTRA-small-v3-modu-ner")
result = ner("삼성전자는 서울에 본사를 두고 있다.")
for entity in result:
    print(f"{entity['word']} → {entity['entity']} ({entity['score']:.2f})")
# 삼성전자 → B-ORG (0.98)
# 서울 → B-LOC (0.99)

config.json:   0%|          | 0.00/1.88k [00:00<?, ?B/s]

c:\Users\human\miniconda3\envs\human-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\human\.cache\huggingface\hub\models--Leo97--KoELECTRA-small-v3-modu-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 56.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/365 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/815k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

삼성전자 → B-OG (0.97)
서울 → B-LC (0.97)


In [14]:
# ④ 마스크 채우기 (MLM 시연)
fill_mask = pipeline("fill-mask", model="klue/bert-base")
result = fill_mask("AI 기술이 [MASK] 발전하고 있다.")
for r in result[:]:
    print(f"{r['token_str']}: {r['score']:.3f}")
# 빠르게: 0.312
# 급속도로: 0.198
# 지속적으로: 0.145

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

c:\Users\human\miniconda3\envs\human-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\human\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

급속히: 0.154
계속: 0.136
날로: 0.136
나날이: 0.129
급격히: 0.105


In [19]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

text = "딥러닝 모델 훈련 방법"
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

with torch.no_grad():
    outputs = model(**inputs)

# print(outputs)

# last_hidden_state: 각 토큰의 문맥화된 표현 - 토큰 임베딩
print(outputs.last_hidden_state.shape)  # [1, seq_len, 768]

# [CLS] 토큰 벡터 = 압축된 문장 전체 표현 (분류 작업에 활용)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)  # [1, 768]

# pooler_output: [CLS]를 Linear+Tanh 통과한 결과 센텐스 임베딩이 된 이 것을 모델에 사용
print(outputs.pooler_output.shape)  # [1, 768]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[   2,  893, 2030, 2944, 4347, 4484, 3826,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 8, 768])
torch.Size([1, 768])
torch.Size([1, 768])
